In [17]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import rand_score
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
import time
import scipy.io
from scipy.spatial.distance import cdist

In [28]:
def coreset(data, m):
    N = data.shape[0]
    
    median = np.median(data, axis=0)
    median = median.reshape(1, -1)
    
    q = cdist(data, median, 'cityblock')
    sum = np.sum(q)
    q = 0.5 * (1.0/N + q/sum)
    q = q.reshape(-1)
    
    samples = np.random.choice(N, m, p=q)
    coreset = data[samples]
    weights = 1.0 / (q[samples] * m)
    
    return coreset, weights 

In [29]:
def kmeans_cost(data, centers, labels):
    cost = 0.0
    for i in range(len(data)):
        distance = np.linalg.norm(data[i] - centers[labels[i]])
        cost += distance
    return cost

In [30]:
def get_results_kmeans(coreset_size, n_clusters, traindata, data_name):
    results = []
    kmeans = KMeans(n_clusters=n_clusters, init="k-means++").fit(traindata)
    optimal_labels = kmeans.predict(traindata)
    cost = kmeans_cost(traindata, kmeans.cluster_centers_, optimal_labels)
    for ssize in coreset_size:
        tic = time.time()
        avg_cost = 0.0
        rand_index = 0.0
        for _ in range(5):
            X_samples, weights = coreset(traindata, ssize)
            kmeans = KMeans(n_clusters=n_clusters, init="k-means++").fit(X_samples, sample_weight=weights)
            labels = kmeans.predict(traindata)
            centers = kmeans.cluster_centers_
            avg_cost += kmeans_cost(traindata, centers, labels)
            rand_index += rand_score(optimal_labels, labels)
        rand_index /= 5
        avg_cost /= 5
        toc = time.time()
        reduction = ((traindata.shape[0] - X_samples.shape[0])/traindata.shape[0])*100
        error = (abs(avg_cost - cost)/cost)*100
        results.append({'Sampling Type': 'Lightweight Coreset (Median)',
                            'Coreset Size': X_samples.shape[0],
                            'Average Cost': avg_cost,
                            'Reduction in Data Size': reduction,
                            'Error': error,
                            'Avg Rand Index': rand_index,
                            'Data': data_name,
                            'Optimal Cost': cost,
                            'Avg Time': (toc - tic)/5,
                            'Clustering Algorithm': 'KMeans++'})
    return results

In [31]:
def kmedoids_cost(data, centroids, labels):
    dist = cdist(data, centroids, 'cityblock')
    total_cost = np.sum(dist[np.arange(len(data)), labels])
    return total_cost

def kmediod(data, weights, k, max_iterations=500):
    data = np.asarray(data)
    
    mins = data.min(axis=0)
    maxs = data.max(axis=0)
    centroids = np.random.rand(k, data.shape[1]) * (maxs - mins) + mins
    
    for _ in range(max_iterations):
        dist = cdist(data, centroids, 'cityblock')
        weighted_dist = dist * weights[:, np.newaxis]
        labels = np.argmin(weighted_dist, axis=1)
        
        for j in range(k):
            cluster = labels == j
            if weights[cluster].sum() > 0:
                centroids[j] = np.average(data[cluster], axis=0, weights=weights[cluster])
            else:
                centroids[j] = np.random.rand(1, data.shape[1]) * (maxs - mins) + mins
    
    return centroids

def predict(data, centroids):
    dist = cdist(data, centroids, 'cityblock')
    labels = np.argmin(dist, axis=1)
    return labels

In [32]:
def get_results_kmediods(coreset_size, n_clusters, traindata, data_name):
    results = []
    centers = kmediod(traindata, np.ones(traindata.shape[0]), n_clusters)
    optimal_labels = predict(traindata, centers)
    cost = kmedoids_cost(traindata, centers, optimal_labels)
    for ssize in coreset_size:
        tic = time.time()
        avg_cost = 0.0
        rand_index = 0.0
        for _ in range(5):
            X_samples, weights = coreset(traindata, ssize)
            centers = kmediod(X_samples, weights, n_clusters)
            labels = predict(traindata, centers)
            avg_cost += kmedoids_cost(traindata, centers, labels)
            rand_index += rand_score(optimal_labels, labels)
        rand_index /= 5
        avg_cost /= 5
        toc = time.time()
        reduction = ((traindata.shape[0] - X_samples.shape[0])/traindata.shape[0])*100
        error = (abs(avg_cost - cost)/cost)*100
        results.append({'Sampling Type': 'Lightweight Coreset (Median)',
                            'Coreset Size': X_samples.shape[0],
                            'Average Cost': avg_cost,
                            'Reduction in Data Size': reduction,
                            'Error': error,
                            'Avg Rand Index': rand_index,
                            'Data': data_name,
                            'Optimal Cost': cost,
                            'Avg Time': (toc - tic)/5,
                            'Clustering Algorithm': 'KMedoids'})
    return results

In [33]:
traindata = pd.DataFrame(pd.read_csv("bio_train.dat", sep="\t", header=None))
traindata = traindata.dropna()
traindata = traindata.drop_duplicates()
traindata = traindata[:10000]

In [34]:
results = get_results_kmeans([100, 500, 1000, 1500, 3500, 5000, 7000, 10000], 50, traindata.values, 'KDD')

C:\Users\pulki\AppData\Roaming\Python\Python311\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\pulki\AppData\Roaming\Python\Python311\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\pulki\AppData\Roaming\Python\Python311\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\pulki\AppData\Roaming\Python\Python311\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The defa

In [35]:
df = pd.read_csv('results.csv')
df = pd.concat([df, pd.DataFrame(results)], ignore_index=True)
df.to_csv('results.csv', index=False)

In [36]:
results = get_results_kmediods([100, 500, 1000, 1500, 3500, 5000, 7000, 10000], 50, traindata.values, 'KDD')

In [37]:
df = pd.read_csv('results.csv')
df = pd.concat([df, pd.DataFrame(results)], ignore_index=True)
df.to_csv('results.csv', index=False)

In [38]:
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=10000, centers=50, n_features=100, random_state=42)

In [39]:
results = get_results_kmeans([100, 500, 1000, 1500, 3500, 5000, 7000, 10000], 50, X, 'Synthetic')

C:\Users\pulki\AppData\Roaming\Python\Python311\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\pulki\AppData\Roaming\Python\Python311\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\pulki\AppData\Roaming\Python\Python311\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\pulki\AppData\Roaming\Python\Python311\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The defa

In [40]:
results

[{'Sampling Type': 'Lightweight Coreset (Median)',
  'Coreset Size': 100,
  'Average Cost': 219880.87794541713,
  'Reduction in Data Size': 99.0,
  'Error': 120.90066263876994,
  'Avg Rand Index': 0.9919876027602761,
  'Data': 'Synthetic',
  'Optimal Cost': 99538.35145573084,
  'Avg Time': 0.1875084400177002,
  'Clustering Algorithm': 'KMeans++'},
 {'Sampling Type': 'Lightweight Coreset (Median)',
  'Coreset Size': 500,
  'Average Cost': 104991.11185150019,
  'Reduction in Data Size': 95.0,
  'Error': 5.478049732614305,
  'Avg Rand Index': 1.0,
  'Data': 'Synthetic',
  'Optimal Cost': 99538.35145573084,
  'Avg Time': 0.2244192600250244,
  'Clustering Algorithm': 'KMeans++'},
 {'Sampling Type': 'Lightweight Coreset (Median)',
  'Coreset Size': 1000,
  'Average Cost': 102168.73331768734,
  'Reduction in Data Size': 90.0,
  'Error': 2.6425813000593523,
  'Avg Rand Index': 1.0,
  'Data': 'Synthetic',
  'Optimal Cost': 99538.35145573084,
  'Avg Time': 0.343049430847168,
  'Clustering Algori

In [41]:
df = pd.read_csv('results.csv')
df = pd.concat([df, pd.DataFrame(results)], ignore_index=True)
df.to_csv('results.csv', index=False)

In [42]:
results = get_results_kmediods([100, 500, 1000, 1500, 3500, 5000, 7000, 10000], 50, X ,'Synthetic')

In [43]:
df = pd.read_csv('results.csv')
df = pd.concat([df, pd.DataFrame(results)], ignore_index=True)
df.to_csv('results.csv', index=False)

In [26]:
mat_data = scipy.io.loadmat('olivettifaces.mat')
traindata = mat_data['faces'].T
traindata = pd.DataFrame(traindata)
traindata.dropna()
traindata.drop_duplicates()
traindata.shape
results = get_results_kmeans([10, 20, 50, 70, 100], 10, traindata.values, 'Face')

C:\Users\pulki\AppData\Roaming\Python\Python311\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


ValueError: 'p' must be 1-dimensional

In [27]:
df = pd.read_csv('results.csv')
df = pd.concat([df, pd.DataFrame(results)], ignore_index=True)
df.to_csv('results.csv', index=False)

NameError: name 'results' is not defined

In [ ]:
results = get_results_kmediods([10, 20, 50, 70, 100], 10, traindata.values, 'Face')

In [ ]:
df = pd.read_csv('results.csv')
df = pd.concat([df, pd.DataFrame(results)], ignore_index=True)
df.to_csv('results.csv', index=False)